In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (Zhao et al.)

This notebook processes and standardizes peptide datasets derived from **Zhao et al.**, which include experimentally annotated **hemolytic** and **toxic** peptide sequences provided as separate Excel files.\
 The goal is to transform these raw sources into clean, non-redundant datasets compatible with downstream machine learning and comparative analyses.

- **Toxic effect / endpoint:** hemolytic, toxic
- **Source:** Zhao et al.
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Reads hemolytic and toxic peptide datasets** from Excel files.
- **Standardizes column names and formats** to a common schema:
  - `sequence`: peptide amino-acid sequence
  - `label`: activity/toxicity annotation
- **Processes each dataset independently** to preserve task-specific annotations.
- **Performs duplicate sequence quality control**:
  - merges identical sequences with consistent labels,
  - identifies and records sequences with conflicting annotations as errors.
- **Builds dataset-level metadata** using the centralized raw-data description spreadsheet.
- **Exports curated datasets** separately for hemolytic and toxic activities, along with error reports.

In [2]:
name_source = "Zhao et al."
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants.
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_hemolytic = (pd.read_excel(f"{PATH_INPUT}/{name_source}/Hemloytic Peptides(seguence).xlsx")
                .rename(columns={"Sequence": "sequence", "Activity": "label"})
                [["sequence", "label"]])
df_hemolytic.shape

(188, 2)

In [4]:
df_toxic = (pd.read_excel(f"{PATH_INPUT}/{name_source}/Toxic Peptides(seguence).xlsx")
                .rename(columns={"Sequence": "sequence", "Activity": "label"})
                [["sequence", "label"]])
df_toxic.shape

(2000, 2)

- Checking duplicates

In [5]:
df_remove_duplicated_hemo, df_errors_hemo, df_unique_hemo = processing_duplicated(df_hemolytic, group_seq="sequence", sort_key="label")
df_full_hemo = pd.concat([df_unique_hemo, df_remove_duplicated_hemo], axis=0)

In [6]:
df_remove_duplicated_tox, df_errors_tox, df_unique_tox = processing_duplicated(df_toxic, group_seq="sequence", sort_key="label")
df_full_tox = pd.concat([df_unique_tox, df_remove_duplicated_tox], axis=0)

In [7]:
df_full_hemolytic = pd.concat([df_unique_hemo, df_remove_duplicated_hemo])
df_full_toxic = pd.concat([df_unique_tox, df_remove_duplicated_tox])
df_full = pd.concat([df_full_hemolytic, df_full_toxic])
df_errors = pd.concat([df_errors_hemo, df_errors_tox])

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
raw_total_sequences = (len(df_hemolytic) + len(df_toxic))

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'Creative Commons Attribution 4.0',
 'year of publication': 2021,
 'last update date': datetime.datetime(2021, 5, 26, 0, 0),
 'download date': Timestamp('2025-06-25 00:00:00'),
 'file format': 'xlsx',
 'peptide property': 'hemolytic, toxic;toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from another DB',
 'repository or server': 'https://www.mdpi.com/1422-0067/22/11/5630/s',
 'publication': 'https://www.mdpi.com/1422-0067/22/11/5630',
 'number_of_raw_sequences': 2188,
 'number_of_sequences_retained': 2186,
 'number_of_positive_sequences': 1093,
 'number_of_negative_sequences': 1093,
 'number_of_erroneous_sequences': 1,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full_hemolytic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_toxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)

df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)